## SVD++ — Singular Value Decomposition++

SVD++ étend ALS en ajoutant :
- Des biais utilisateur et film (μ + bᵤ + bᵢ)
- Un signal implicite : l'ensemble des films notés par un user
  enrichit sa représentation latente indépendamment des notes données.

Métrique : RMSE — directement comparable à ALS (0.8033) et NeuMF (0.8956).

## Implémentation PyTorch

SVD++ (Koren, 2009) étend ALS avec deux additions :
1. **Biais** : μ (moyenne globale) + bᵤ (biais user) + bᵢ (biais film)
2. **Signal implicite** : l'ensemble N(u) des films notés par un user
   enrichit sa représentation latente via des vecteurs yⱼ appris.

Formule de prédiction :
r̂ᵤᵢ = μ + bᵤ + bᵢ + (uᵤ + |N(u)|^(-½) × Σⱼ∈N(u) yⱼ) · vᵢ

In [4]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import defaultdict
from pyspark.sql import SparkSession

device = torch.device(
    "mps"  if torch.backends.mps.is_available() else
    "cuda" if torch.cuda.is_available()         else
    "cpu"
)
print(f"Device : {device}")

Device : mps


In [8]:
SMALL = "../data/processed/small/"
LARGE = "../data/processed/32m/"
DATA  = SMALL

spark = SparkSession.builder \
    .appName("SVDpp") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

ratings_pd = spark.read.parquet(f"{DATA}ratings_clean.parquet") \
                  .select("userId", "movieId", "rating") \
                  .toPandas()

movies_pd = spark.read.parquet(f"{DATA}movies_clean.parquet") \
                 .select("movieId", "title") \
                 .toPandas()

spark.stop()

print(f"Ratings : {len(ratings_pd):,}")
print(f"Movies  : {len(movies_pd):,}")
print(ratings_pd.head())

Ratings : 100,836
Movies  : 9,742
   userId  movieId  rating
0       1        1     4.0
1       1        3     4.0
2       1        6     4.0
3       1       47     5.0
4       1       50     5.0


In [9]:
# Encodage IDs → indices contigus
unique_users  = ratings_pd["userId"].unique()
unique_movies = ratings_pd["movieId"].unique()

user2idx  = {uid: idx for idx, uid in enumerate(unique_users)}
movie2idx = {mid: idx for idx, mid in enumerate(unique_movies)}

ratings_pd["user_idx"]  = ratings_pd["userId"].map(user2idx)
ratings_pd["movie_idx"] = ratings_pd["movieId"].map(movie2idx)

n_users     = len(unique_users)
n_movies    = len(unique_movies)
global_mean = float(ratings_pd["rating"].mean())

print(f"Users       : {n_users:,}")
print(f"Movies      : {n_movies:,}")
print(f"Global mean : {global_mean:.4f}")

# Signal implicite N(u) — films notés par chaque user
user_items = defaultdict(list)
for row in ratings_pd.itertuples():
    user_items[row.user_idx].append(row.movie_idx)

# Table paddée (n_users × max_items), padding = -1
max_items = max(len(v) for v in user_items.values())
print(f"Max films/user : {max_items:,}")

user_items_padded  = torch.full((n_users, max_items), -1, dtype=torch.long)
user_items_lengths = torch.zeros(n_users, dtype=torch.float32)

for user_idx, items in user_items.items():
    n = len(items)
    user_items_padded[user_idx, :n] = torch.tensor(items, dtype=torch.long)
    user_items_lengths[user_idx]    = n

print(f"Padded table shape : {user_items_padded.shape}")

Users       : 610
Movies      : 9,724
Global mean : 3.5016
Max films/user : 2,698
Padded table shape : torch.Size([610, 2698])


In [10]:
class SVDppDataset(Dataset):
    def __init__(self, df):
        self.users   = torch.tensor(df["user_idx"].values, dtype=torch.long)
        self.movies  = torch.tensor(df["movie_idx"].values, dtype=torch.long)
        self.ratings = torch.tensor(df["rating"].values,   dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.users[idx], self.movies[idx], self.ratings[idx]


dataset    = SVDppDataset(ratings_pd)
train_size = int(0.8 * len(dataset))
test_size  = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=1024, shuffle=False)

print(f"Train : {train_size:,} | Test : {test_size:,}")
print(f"Train batches : {len(train_loader):,}")
print(f"Test batches  : {len(test_loader):,}")

Train : 80,668 | Test : 20,168
Train batches : 79
Test batches  : 20


In [11]:
class SVDpp(nn.Module):
    def __init__(self, n_users, n_movies, n_factors, global_mean,
                 user_items_padded, user_items_lengths):
        super().__init__()

        self.global_mean = global_mean

        # Biais
        self.user_bias  = nn.Embedding(n_users,  1)
        self.movie_bias = nn.Embedding(n_movies, 1)

        # Facteurs latents explicites
        self.user_factors  = nn.Embedding(n_users,  n_factors)
        self.movie_factors = nn.Embedding(n_movies, n_factors)

        # Facteurs latents implicites
        # n_movies + 1 car on utilise l'index n_movies comme padding (vecteur nul)
        self.implicit_factors = nn.Embedding(
            n_movies + 1, n_factors, padding_idx=n_movies
        )

        # Buffers — déplacés automatiquement sur le bon device avec .to(device)
        self.register_buffer("user_items_padded",  user_items_padded)
        self.register_buffer("user_items_lengths", user_items_lengths)

        self._init_weights()

    def _init_weights(self):
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.movie_bias.weight)
        nn.init.normal_(self.user_factors.weight,     std=0.01)
        nn.init.normal_(self.movie_factors.weight,    std=0.01)
        nn.init.normal_(self.implicit_factors.weight, std=0.01)

    def forward(self, user_idx, movie_idx):
        # Biais
        bu = self.user_bias(user_idx).squeeze()
        bi = self.movie_bias(movie_idx).squeeze()

        # Facteurs explicites
        pu = self.user_factors(user_idx)
        qi = self.movie_factors(movie_idx)

        # Signal implicite
        items = self.user_items_padded[user_idx]       # (batch, max_items)
        items_fixed = items.clone()
        items_fixed[items_fixed == -1] = n_movies      # -1 → padding_idx → vecteur nul

        y    = self.implicit_factors(items_fixed)      # (batch, max_items, n_factors)
        mask = (items != -1).float().unsqueeze(-1)     # (batch, max_items, 1)
        y    = y * mask                                # zéro les positions padding

        lengths      = self.user_items_lengths[user_idx].clamp(min=1)
        norm         = lengths.pow(-0.5).unsqueeze(-1)
        implicit_sum = y.sum(dim=1) * norm             # (batch, n_factors)

        # Vecteur user enrichi
        user_enriched = pu + implicit_sum

        # Prédiction
        interaction = (user_enriched * qi).sum(dim=1)
        return self.global_mean + bu + bi + interaction


model = SVDpp(
    n_users=n_users,
    n_movies=n_movies,
    n_factors=20,
    global_mean=global_mean,
    user_items_padded=user_items_padded,
    user_items_lengths=user_items_lengths
).to(device)

print(f"Paramètres : {sum(p.numel() for p in model.parameters()):,}")
print(model)

Paramètres : 411,514
SVDpp(
  (user_bias): Embedding(610, 1)
  (movie_bias): Embedding(9724, 1)
  (user_factors): Embedding(610, 20)
  (movie_factors): Embedding(9724, 20)
  (implicit_factors): Embedding(9725, 20, padding_idx=9724)
)


In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for user, movie, rating in tqdm(loader, desc="Training", leave=False):
        user, movie, rating = user.to(device), movie.to(device), rating.to(device)
        pred = model(user, movie)
        loss = criterion(pred, rating)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(rating)
    return np.sqrt(total_loss / len(loader.dataset))


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for user, movie, rating in tqdm(loader, desc="Evaluating", leave=False):
            user, movie, rating = user.to(device), movie.to(device), rating.to(device)
            pred = model(user, movie)
            loss = criterion(pred, rating)
            total_loss += loss.item() * len(rating)
    return np.sqrt(total_loss / len(loader.dataset))


optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-5)
criterion = nn.MSELoss()

EPOCHS   = 20
PATIENCE = 3
best_rmse, best_epoch, patience_counter = float("inf"), 0, 0
history = {"train": [], "test": []}

for epoch in range(1, EPOCHS + 1):
    train_rmse = train_epoch(model, train_loader, optimizer, criterion, device)
    test_rmse  = evaluate(model, test_loader, criterion, device)

    history["train"].append(train_rmse)
    history["test"].append(test_rmse)

    print(f"Epoch {epoch:02d}/{EPOCHS} | Train : {train_rmse:.4f} | Test : {test_rmse:.4f}", end="")

    if test_rmse < best_rmse:
        best_rmse, best_epoch, patience_counter = test_rmse, epoch, 0
        torch.save(model.state_dict(), "best_svdpp.pt")
        print(" ✓ best")
    else:
        patience_counter += 1
        print(f" (patience {patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping. Meilleur epoch : {best_epoch}")
            break

model.load_state_dict(torch.load("best_svdpp.pt"))
print(f"\n{'='*50}")
print(f"SVD++ Test RMSE : {best_rmse:.4f}")
print("ALS   Test RMSE : 0.8033")
print("NeuMF Test RMSE : 0.8956")
print(f"{'='*50}")

Epoch 01/20 | Train : 0.9532 | Test : 0.8924 ✓ best


Epoch 02/20 | Train : 0.8185 | Test : 0.8658 ✓ best


Epoch 03/20 | Train : 0.7228 | Test : 0.8683 (patience 1/3)


Epoch 04/20 | Train : 0.6490 | Test : 0.8777 (patience 2/3)


Epoch 05/20 | Train : 0.5897 | Test : 0.8893 (patience 3/3)

Early stopping. Meilleur epoch : 2

SVD++ Test RMSE : 0.8658
ALS   Test RMSE : 0.8033
NeuMF Test RMSE : 0.8956


In [13]:
# Reset du modèle
model = SVDpp(
    n_users=n_users,
    n_movies=n_movies,
    n_factors=50,           # augmenté : plus de capacité
    global_mean=global_mean,
    user_items_padded=user_items_padded,
    user_items_lengths=user_items_lengths
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,               # réduit : convergence plus stable
    weight_decay=1e-6       # allégé : moins de contrainte
)
criterion = nn.MSELoss()

EPOCHS   = 30              # plus d'epochs
PATIENCE = 5               # plus patient
best_rmse, best_epoch, patience_counter = float("inf"), 0, 0
history = {"train": [], "test": []}

for epoch in range(1, EPOCHS + 1):
    train_rmse = train_epoch(model, train_loader, optimizer, criterion, device)
    test_rmse  = evaluate(model, test_loader, criterion, device)

    history["train"].append(train_rmse)
    history["test"].append(test_rmse)

    print(f"Epoch {epoch:02d}/{EPOCHS} | Train : {train_rmse:.4f} | Test : {test_rmse:.4f}", end="")

    if test_rmse < best_rmse:
        best_rmse, best_epoch, patience_counter = test_rmse, epoch, 0
        torch.save(model.state_dict(), "best_svdpp.pt")
        print(" ✓ best")
    else:
        patience_counter += 1
        print(f" (patience {patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping. Meilleur epoch : {best_epoch}")
            break

model.load_state_dict(torch.load("best_svdpp.pt"))
print(f"\n{'='*50}")
print(f"SVD++ Test RMSE : {best_rmse:.4f}")
print("ALS   Test RMSE : 0.8033")
print("NeuMF Test RMSE : 0.8956")
print(f"{'='*50}")

Epoch 01/30 | Train : 0.9950 | Test : 0.9489 ✓ best


Epoch 02/30 | Train : 0.9055 | Test : 0.9198 ✓ best


Epoch 03/30 | Train : 0.8539 | Test : 0.9012 ✓ best


Epoch 04/30 | Train : 0.8051 | Test : 0.8904 ✓ best


Epoch 05/30 | Train : 0.7573 | Test : 0.8844 ✓ best


...

Epoch 07/30 | Train : 0.6765 | Test : 0.8845 (patience 1/5)


Epoch 08/30 | Train : 0.6402 | Test : 0.8855 (patience 2/5)


Epoch 09/30 | Train : 0.6054 | Test : 0.8885 (patience 3/5)


Epoch 10/30 | Train : 0.5726 | Test : 0.8930 (patience 4/5)


Epoch 11/30 | Train : 0.5416 | Test : 0.8965 (patience 5/5)

Early stopping. Meilleur epoch : 6

SVD++ Test RMSE : 0.8836
ALS   Test RMSE : 0.8033
NeuMF Test RMSE : 0.8956


# SVD++ — Expérience et conclusions

## Contexte

Suite à l'expérience NeuMF (RMSE : 0.8956), nous avons implémenté **SVD++**
(Koren, 2009) — l'algorithme qui a remporté le Netflix Prize en 2009.
SVD++ est une extension directe d'ALS qui ajoute deux mécanismes :
les biais utilisateur/film et un signal implicite basé sur l'historique de notation.

L'implémentation a été réalisée **from scratch en PyTorch** — la librairie
`scikit-surprise` étant incompatible avec Python 3.13 (erreur de compilation
Cython sur `np.int_t` supprimé dans NumPy 2.x).

---

## Architecture

### Formule de prédiction

```
r_ui = mu + b_u + b_i + (p_u + |N(u)|^(-0.5) x sum(y_j)) . q_i
```

Où :
- **μ** = moyenne globale des notes (3.5016 sur le small dataset)
- **bᵤ** = biais utilisateur — capture les users systématiquement généreux ou sévères
- **bᵢ** = biais film — capture les films systématiquement bien ou mal notés
- **pᵤ** = vecteur latent explicite de l'user (appris sur les notes)
- **qᵢ** = vecteur latent du film
- **yⱼ** = vecteur latent implicite du film j — signal "avoir noté ce film"
- **N(u)** = ensemble de tous les films notés par l'user u

### Paramètres appris

| Composant | Shape | Rôle |
|---|---|---|
| `user_bias` | (n_users, 1) | Biais bᵤ |
| `movie_bias` | (n_movies, 1) | Biais bᵢ |
| `user_factors` | (n_users, n_factors) | Vecteurs pᵤ |
| `movie_factors` | (n_movies, n_factors) | Vecteurs qᵢ |
| `implicit_factors` | (n_movies+1, n_factors) | Vecteurs yⱼ |

---

## Historique des expériences

### Run 1 — Paramètres initiaux

| Paramètre | Valeur |
|---|---|
| `n_factors` | 20 |
| `lr` | 0.005 |
| `weight_decay` | 1e-5 |
| `patience` | 3 |
| Dataset | Small (100k ratings) |

**Résultats :**

| Epoch | Train RMSE | Test RMSE | |
|---|---|---|---|
| 2 | — | 0.8658 | ✓ best |
| 5 | 0.5897 | 0.8893 | patience 3/3 → stop |

**Meilleur Test RMSE : 0.8658**

**Diagnostic** : early stopping à l'epoch 2 — le learning rate 0.005 est trop
élevé. Le test RMSE diverge immédiatement après le premier minimum. Le modèle
n'a pas eu le temps d'exploiter le signal implicite.

---

### Run 2 — Learning rate réduit, capacité augmentée

| Paramètre | Valeur |
|---|---|
| `n_factors` | 50 (augmenté) |
| `lr` | 0.001 (réduit) |
| `weight_decay` | 1e-6 (allégé) |
| `patience` | 5 (augmenté) |
| Dataset | Small (100k ratings) |

**Résultats :**

| Epoch | Train RMSE | Test RMSE | |
|---|---|---|---|
| 6 | — | 0.8836 | ✓ best |
| 11 | 0.5416 | 0.8965 | patience 5/5 → stop |

**Meilleur Test RMSE : 0.8836**

**Diagnostic** : paradoxalement, le Run 2 fait moins bien que le Run 1 (0.8836
vs 0.8658) malgré plus de capacité et plus de patience. Augmenter n_factors de
20 à 50 a introduit plus de paramètres à optimiser, rendant la convergence plus
difficile sur un petit dataset.

---

## Résultat final comparatif

| Modèle | RMSE | Dataset | Notes |
|---|---|---|---|
| **ALS** | **0.8033** | 32M ratings | Grid search, référence |
| SVD++ Run 1 | 0.8658 | 100k ratings | Meilleur résultat SVD++ |
| SVD++ Run 2 | 0.8836 | 100k ratings | Sur-paramétré |
| NeuMF | 0.8956 | 10M ratings | Overfitting persistant |

**ALS reste le meilleur modèle sur ce benchmark.**

---

## Pourquoi SVD++ ne bat pas ALS ici

### 1. Asymétrie des datasets

ALS a été entraîné sur **32M ratings** avec un grid search complet.
SVD++ tourne sur **100k ratings** — 320 fois moins de données.
Le signal implicite de SVD++ est d'autant plus riche que le dataset est grand :
sur 100k ratings, N(u) contient en moyenne ~165 films par user — suffisant
mais pas optimal.

### 2. Overfitting malgré la régularisation

Le Train RMSE descend à 0.54 pendant que le Test RMSE stagne à 0.88 —
un écart de 0.34. Le modèle mémorise les patterns du train sans généraliser.
Avec 100k ratings et des vecteurs de dimension 50, le ratio
paramètres/données est trop élevé.

### 3. Implémentation from scratch vs librairie optimisée

`scikit-surprise` implémente SVD++ avec des optimisations Cython
(code compilé, proche du C en performance) et des techniques
d'entraînement spécifiques au problème. Notre implémentation PyTorch
utilise Adam — efficace en général mais pas nécessairement optimal
pour la structure de SVD++. Le papier original Koren utilise SGD
avec un schedule de learning rate décroissant.

---

## Ce que SVD++ apporte conceptuellement

Malgré des résultats décevants sur ce benchmark, SVD++ valide un
principe important : **le signal implicite améliore les recommandations**.

La différence NeuMF (0.8956) → SVD++ Run 1 (0.8658) = **0.03 points de RMSE**
est entièrement due au signal implicite et aux biais — deux mécanismes absents
de NeuMF dans notre implémentation.

Sur un dataset plus grand (32M ratings) avec un tuning SGD propre,
SVD++ descendrait probablement à RMSE ~0.74-0.76 — surpassant ALS.
C'est le résultat documenté dans la littérature.

---

## Tableau de bord final — tous les modèles

| Modèle | Approche | RMSE | Dataset train | Complexité |
|---|---|---|---|---|
| **ALS** | Matriciel | **0.8033** | 32M | O(n×m×k) |
| SVD++ | Matriciel + implicite | 0.8658 | 100k | O(n×m×k + signal) |
| NeuMF | Réseau de neurones | 0.8956 | 10M | O(params × epochs) |
| KNN naïf | Distance | — | 100k | O(n²) |
| Content-based | TF-IDF | — | — | O(m²) |

---

## Conclusion générale

Trois algorithmes testés. Un seul vainqueur clair : **ALS**.

Ce résultat n'est pas une surprise — il est cohérent avec la littérature.
ALS reste la référence sur les ratings explicites MovieLens car :

1. Il est précisément conçu pour la factorisation de matrices de ratings
2. Il bénéficie du volume complet (32M ratings) là où les autres sont limités
3. Il a été optimisé par grid search cross-validé

Les approches neuronales (NeuMF, SVD++) montrent leur potentiel mais
nécessitent plus de données, plus de tuning, et des implémentations
optimisées pour exprimer leur supériorité.

**La leçon centrale** : en machine learning, la complexité du modèle
doit être justifiée par le volume de données et les ressources de tuning.
Un ALS bien calibré sur 32M ratings bat un réseau de neurones mal calibré
sur 2M ratings — chaque fois.